In [2]:
#-- Import library. --#
import os
import re
import pandas as pd

In [17]:
input_file = '../Raw/Traffic_Count_Locations_with_LONG_LAT.csv'
output_dir = '../Processed/'

In [20]:
df = pd.read_csv(input_file, encoding='utf-8-sig', low_memory=False)
df.columns = [re.sub(r'[^a-zA-Z0-9_]', '', col).lower() for col in df.columns]
print("Lists of column labels:", df.columns.tolist())

Lists of column labels: ['x', 'y', 'fid', 'objectid', 'tfm_id', 'tfm_desc', 'tfm_typ_de', 'movement_t', 'site_desc', 'road_nbr', 'declared_r', 'local_road', 'data_src_c', 'data_sourc', 'time_categ', 'year_since', 'last_year', 'aadt_allve', 'aadt_truck', 'per_trucks']


In [22]:
target_col = 'local_road'

if target_col in df.columns:
    # Lọc bỏ dòng lỗi
    df = df.dropna(subset=[target_col, 'x', 'y'])
    df = df[(df['x'] != '0') & (df['y'] != '0')] # x, y giờ cũng là lowercase

    # Định nghĩa nhóm chữ cái (vẫn dùng chữ hoa để check vì ta sẽ .upper() ký tự đầu)
    alphabet_groups = {
        'a_d': ['A', 'B', 'C', 'D'],
        'e_h': ['E', 'F', 'G', 'H'],
        'i_l': ['I', 'J', 'K', 'L'],
        'm_p': ['M', 'N', 'O', 'P'],
        'q_t': ['Q', 'R', 'S', 'T'],
        'u_z': ['U', 'V', 'W', 'X', 'Y', 'Z']
    }

    print("\nProcessing split data...")
    for group_name, letters in alphabet_groups.items():
        # Lấy ký tự đầu tiên của tên đường và viết hoa để so sánh với alphabet_groups
        mask = df[target_col].str[0].str.upper().isin(letters)
        group_df = df[mask]
        
        if not group_df.empty:
            save_path = os.path.join(output_dir, f"roads_{group_name}.csv")
            group_df.to_csv(save_path, index=False)
            print(f"-> Saved: Roads_{group_name}.csv")

    # Nhóm còn lại (số/ký tự lạ)
    others_mask = ~df[target_col].str[0].str.upper().str.isalpha()
    others_df = df[others_mask]
    if not others_df.empty:
        others_df.to_csv(os.path.join(output_dir, "roads_others.csv"), index=False)
        print("-> Saved: roads_others.csv")
else:
    print(f"No column find '{target_col}'!")

print("\n--- FInish: Data has been lowercase and split ---")


Processing split data...
-> Saved: Roads_a_d.csv
-> Saved: Roads_e_h.csv
-> Saved: Roads_i_l.csv
-> Saved: Roads_m_p.csv
-> Saved: Roads_q_t.csv
-> Saved: Roads_u_z.csv
-> Saved: roads_others.csv

--- FInish: Data has been lowercase and split ---
